# 02c — EDA: clustering jerárquico y PCA exploratorio (edad × sexo)

**Sub-fase 2c — CRISP-DM: Data Understanding (extensión no supervisada)** ·
Rama `feature/fase-2c-eda-clustering`, abierta a pedido del usuario tras cerrar
el Loop C (`02_eda.ipynb`).

**Objetivo.** Explorar si un clustering jerárquico sobre el perfil bioquímico,
estratificado por banda de edad y sexo, revela (a) patrones sugerentes de
heterogeneidad etiológica/severidad, y (b) puntos de corte candidatos para
`Sgpt` (ALT) comparables contra los umbrales de literatura ya adoptados
(`docs/adr/0004-umbrales-referencia-sexo-especificos.md`).

**⚠️ Alcance y límites, acordados con el usuario antes de escribir código
(ver `AGENTS.md`, checkpoint 2026-07-28):**

1. Esto es **exploración de hipótesis, no un modelo**. No se entrena ningún
   clasificador ni se calculan métricas de clasificación — sigue vigente el
   límite del PRD (§2.3) para las Fases 0–3.
2. Cualquier "punto de corte candidato" que salga de acá es **señal
   exploratoria derivada de este dataset**, explícitamente **no validada
   clínicamente**. Nunca reemplaza los umbrales de Prati/ACG/AASLD del ADR
   0004 — en el peor caso los contradice, y ahí se explica por qué en vez de
   descartar la contradicción.
3. Notebook separado de `02_eda.ipynb` a propósito: mezclar clustering
   exploratorio con la EDA que responde a la rúbrica (T1–T5) diluiría cuál
   celda responde a qué.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import matplotlib
matplotlib.use("Agg")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram
from sklearn.preprocessing import StandardScaler

from src.config import (
    NUMERIC_COLS,
    LIVER_AXES,
    ALT_ULN_BY_SEX,
    INK_PRIMARY,
    INK_SECONDARY,
    GRID_COLOR,
    SURFACE,
    SEX_COLORS,
    DPI,
)
from src.utils import (
    load_raw_data,
    save_figure,
    flag_biochemical_violations,
    assign_age_sex_stratum,
    hierarchical_cluster_cut,
    de_ritis_ratio,
)

plt.rcParams.update({
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "axes.edgecolor": INK_SECONDARY,
    "axes.labelcolor": INK_PRIMARY,
    "text.color": INK_PRIMARY,
    "xtick.color": INK_SECONDARY,
    "ytick.color": INK_SECONDARY,
    "grid.color": GRID_COLOR,
})

## Preparación — exclusiones de calidad conocidas y estratificación

**Exclusiones.** Este notebook trabaja sobre datos **crudos, sin imputar**
(la imputación formal es tarea de la Fase 3, todavía sin código — ver
`feature/fase-3-preprocessing`). Pero incluir filas con violaciones
bioquímicas ya documentadas inyectaría ruido conocido en un análisis que
busca justamente patrones de daño real:

- **4 filas con `A/G Ratio` faltante** (Q2) — `StandardScaler` no acepta
  `NaN`.
- **3 filas con `DB > TB`** (Q8, Loop C) — violación de plausibilidad ya
  documentada; su `TB`/`DB` no son confiables para ningún análisis.

Se excluyen ambas (7 filas, n efectivo 576) **solo para este notebook
exploratorio** — la Fase 3 decide su tratamiento definitivo (imputación) de
forma independiente, ver `AGENTS.md`.

In [2]:
df = load_raw_data()

missing_ag = df["A/G Ratio"].isna()
db_gt_tb = flag_biochemical_violations(df)["db_gt_tb"]
excluded = missing_ag | db_gt_tb

print(f"Filas excluidas: {excluded.sum()} (A/G faltante: {missing_ag.sum()}, DB>TB: {db_gt_tb.sum()})")
print(f"n para clustering: {(~excluded).sum()}")

work = df[~excluded].copy()
work["Stratum"] = assign_age_sex_stratum(work)
work["Stratum"].value_counts()

Filas excluidas: 7 (A/G faltante: 4, DB>TB: 3)
n para clustering: 576


Stratum
40-59 · Male          156
18-39 · Male          150
60-120 · Male         113
40-59 · Female         65
18-39 · Female         47
0-17 · ambos sexos     25
60-120 · Female        20
Name: count, dtype: int64

**Interpretación.** 7 estratos: las 3 bandas adultas (18–39, 40–59,
60–120) se dividen por sexo porque al menos un sexo en cada una supera 30
casos. La banda pediátrica (0–17, n=25: 7 mujeres/18 hombres) queda como un
solo grupo mixto — dividirla habría dejado sub-muestras de menos de 20
personas, insuficientes para que un dendrograma refleje algo más que ruido
individual (`MIN_STRATUM_SIZE_FOR_SEX_SPLIT` en `src/config.py`).
`60-120 · Female` (n=20) sí queda separado por sexo porque el otro sexo de
esa banda supera el umbral, pero su propio tamaño es chico — se interpreta
con ese caveat explícito más abajo, no se descarta.

## Experimento 1 — clustering sobre las 9 variables completas

Primer intento, tal como se acordó: las 9 variables numéricas
(`NUMERIC_COLS`), estandarizadas *dentro de cada estrato* (para comparar la
posición relativa de cada persona contra su propio grupo de referencia
edad-sexo, no contra la población general), clustering jerárquico (enlace de
Ward) y corte por el mayor salto de distancia de fusión
(`hierarchical_cluster_cut`, sin fijar `k` a mano).

In [3]:
RNG_LABEL_ORDER = ["0-17 · ambos sexos", "18-39 · Female", "18-39 · Male",
                   "40-59 · Female", "40-59 · Male",
                   "60-120 · Female", "60-120 · Male"]

exp1_results = {}
exp1_summary_rows = []

for stratum in RNG_LABEL_ORDER:
    sub = work.loc[work["Stratum"] == stratum].copy()
    X = StandardScaler().fit_transform(sub[NUMERIC_COLS])
    Z, labels, cut, k = hierarchical_cluster_cut(X)
    sub["Cluster"] = labels
    exp1_results[stratum] = {"sub": sub, "Z": Z, "k": k}

    # cluster de "severidad": el que tiene la mediana de TB mas alta
    # (excrecion biliar es el eje menos ruidoso para identificar derangement)
    tb_by_cluster = sub.groupby("Cluster")["TB"].median().sort_values(ascending=False)
    severe_cluster = tb_by_cluster.index[0]
    severe = sub[sub["Cluster"] == severe_cluster]
    rest = sub[sub["Cluster"] != severe_cluster]

    exp1_summary_rows.append({
        "Estrato": stratum,
        "n": len(sub),
        "k": k,
        "n_cluster_severo": len(severe),
        "TB_mediana_severo": round(severe["TB"].median(), 1),
        "TB_mediana_resto": round(rest["TB"].median(), 1),
        "pct_diagnosticado_severo": round(100 * (severe["Selector"] == 1).mean(), 1),
        "pct_diagnosticado_resto": round(100 * (rest["Selector"] == 1).mean(), 1),
    })

exp1_summary = pd.DataFrame(exp1_summary_rows).set_index("Estrato")
exp1_summary

,n,k,n_cluster_severo,TB_mediana_severo,TB_mediana_resto,pct_diagnosticado_severo,pct_diagnosticado_resto
Estrato,,,,,,,
0-17 · ambos sexos,25,2,1,27.2,0.8,100.0,45.8
18-39 · Female,47,4,1,16.7,0.8,100.0,52.2
18-39 · Male,150,2,19,15.9,1.0,100.0,64.9
40-59 · Female,65,5,4,23.2,0.9,100.0,73.8
40-59 · Male,156,3,18,18.0,1.0,100.0,76.1
60-120 · Female,20,4,2,3.6,0.8,100.0,50.0
60-120 · Male,113,4,12,13.2,1.3,100.0,74.3


In [4]:
fig, axes = plt.subplots(4, 2, figsize=(13, 16))
axes = axes.ravel()
for ax, stratum in zip(axes, RNG_LABEL_ORDER):
    Z = exp1_results[stratum]["Z"]
    dendrogram(Z, ax=ax, no_labels=True, color_threshold=0,
               above_threshold_color=INK_SECONDARY)
    ax.set_title(f"{stratum} (n={exp1_results[stratum]['sub'].shape[0]}, "
                 f"k={exp1_results[stratum]['k']})", fontsize=10)
    ax.set_ylabel("distancia (Ward)")
axes[-1].axis("off")
fig.suptitle("Experimento 1 — dendrogramas sobre las 9 variables, por estrato edad-sexo",
             y=1.0, fontsize=12)
fig.tight_layout()
save_figure(fig, "fase2c_exp1_dendrogramas.png")
plt.show()

C:\Users\harrison.tutalcha\AppData\Local\Temp\ipykernel_9188\3282328835.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretación — aparece una "firma de severidad", no un punto de
corte.**

En **todas las bandas adultas** (no en la pediátrica ni en `60-120 · Female`,
ver caveat abajo) aparece el mismo patrón: un cluster **chico**, con
mediana de `TB` muy por encima del rango de referencia (0.3–1.2 mg/dL) y con
un porcentaje de diagnóstico positivo mucho más alto que el resto —en varios
casos, 100%. El resto de las personas —incluyendo casos sanos y casos
diagnosticados pero leves— queda en un cluster grande sin separarse entre sí.

Ejemplo concreto, hombres 18–39 (`n=150`): el cluster severo tiene `n=19`,
mediana `TB=15.9` (13× el límite superior normal), 100% diagnosticado. El
resto (`n=131`) tiene mediana `TB=1.0`, 65% diagnosticado — sano y enfermo
leve mezclados.

**Por qué esto no da un "corte candidato" utilizable:** el clustering separa
por el **perfil de 9 variables combinado**, no por un eje único. Al proyectar
esos clusters sobre `Sgpt` (ALT) solamente, los rangos **se superponen**
(el cluster "resto" llega hasta valores altos, el cluster "severo" a veces
baja a valores moderados) — un punto de corte univariado ahí sería
inventado, no derivado de una separación real.

## Experimento 2 — clustering restringido al eje de daño celular (`Sgpt`, `Sgot`)

Pedido explícito del usuario: repetir el clustering usando solo las 2
variables del eje de daño celular (`LIVER_AXES["dano_celular"]`), para ver
si reducir la dimensionalidad resuelve un corte más fino entre "sano" y
"levemente alterado" — justo el rango donde operan los umbrales de ALT de la
literatura.

Se prueban **dos variantes**: sobre los valores crudos, y sobre
`log1p(valor)` antes de estandarizar. La segunda variante no es arbitraria:
T4 (`02_eda.ipynb`) ya documentó que `Sgpt`/`Sgot` tienen una distribución
muy asimétrica con cola larga — el logaritmo es la transformación estándar
para ese tipo de variable de laboratorio antes de calcular distancias.

In [5]:
damage_cols = LIVER_AXES["dano_celular"]
exp2_results = {}
exp2_summary_rows = []

for stratum in RNG_LABEL_ORDER:
    sub = work.loc[work["Stratum"] == stratum].copy()

    X_log = StandardScaler().fit_transform(np.log1p(sub[damage_cols]))
    Z_log, labels_log, cut_log, k_log = hierarchical_cluster_cut(X_log)
    sub["ClusterLog"] = labels_log
    exp2_results[stratum] = {"sub": sub, "Z": Z_log, "k": k_log}

    # cluster de menor mediana de Sgpt = candidato a "linea de base" de este estrato
    medians = sub.groupby("ClusterLog")["Sgpt"].median().sort_values()
    low_cluster = medians.index[0]
    low_vals = sub.loc[sub["ClusterLog"] == low_cluster, "Sgpt"]
    rest_vals = sub.loc[sub["ClusterLog"] != low_cluster, "Sgpt"]

    overlap = low_vals.max() >= rest_vals.min() if len(rest_vals) else np.nan
    candidate_cutoff = np.nan if overlap or not len(rest_vals) else (low_vals.max() + rest_vals.min()) / 2

    sex = sub["Gender"].iloc[0] if sub["Gender"].nunique() == 1 else None
    lit_uln = ALT_ULN_BY_SEX.get(sex, np.nan)

    exp2_summary_rows.append({
        "Estrato": stratum,
        "n": len(sub),
        "k": k_log,
        "n_cluster_base": len(low_vals),
        "Sgpt_max_base": low_vals.max(),
        "Sgpt_min_resto": rest_vals.min() if len(rest_vals) else np.nan,
        "rangos_se_superponen": overlap,
        "corte_candidato": round(candidate_cutoff, 1) if not np.isnan(candidate_cutoff) else np.nan,
        "ULN_literatura": lit_uln,
    })

exp2_summary = pd.DataFrame(exp2_summary_rows).set_index("Estrato")
exp2_summary

,n,k,n_cluster_base,Sgpt_max_base,Sgpt_min_resto,rangos_se_superponen,corte_candidato,ULN_literatura
Estrato,,,,,,,,
0-17 · ambos sexos,25,2,20,58,131,False,94.5,NaN
18-39 · Female,47,2,40,90,70,True,NaN,19.0
18-39 · Male,150,3,75,78,24,True,NaN,30.0
40-59 · Female,65,2,57,96,110,False,103.0,19.0
40-59 · Male,156,2,118,88,39,True,NaN,30.0
60-120 · Female,20,2,17,27,42,False,34.5,19.0
60-120 · Male,113,2,86,116,42,True,NaN,30.0


In [6]:
fig, axes = plt.subplots(4, 2, figsize=(13, 16))
axes = axes.ravel()
for ax, stratum in zip(axes, RNG_LABEL_ORDER):
    Z = exp2_results[stratum]["Z"]
    dendrogram(Z, ax=ax, no_labels=True, color_threshold=0,
               above_threshold_color=INK_SECONDARY)
    ax.set_title(f"{stratum} (n={exp2_results[stratum]['sub'].shape[0]}, "
                 f"k={exp2_results[stratum]['k']})", fontsize=10)
    ax.set_ylabel("distancia (Ward, log1p)")
axes[-1].axis("off")
fig.suptitle("Experimento 2 — dendrogramas sobre Sgpt+Sgot (log1p), por estrato edad-sexo",
             y=1.0, fontsize=12)
fig.tight_layout()
save_figure(fig, "fase2c_exp2_dendrogramas.png")
plt.show()

C:\Users\harrison.tutalcha\AppData\Local\Temp\ipykernel_9188\2799786700.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretación — reducir a 2 variables no resuelve el corte, y la
razón es más interesante que "hay que probar otra transformación".**

La tabla de arriba (`rangos_se_superponen`) muestra que en la mayoría de los
estratos los rangos **siguen superpuestos** incluso restringiendo a solo
`Sgpt`/`Sgot` con `log1p`. Donde sí queda un hueco entre clusters (sin
superposición), el "corte candidato" resultante sigue siendo **varias veces
más alto** que el umbral de literatura (decenas a un centenar de U/L, contra
19–30 U/L de Prati/ACG/AASLD).

**La razón no es un problema de método — es la composición de la muestra.**
El ILPD es una cohorte de **hospital** (71% de los 583 pacientes ya tienen
diagnóstico positivo) — no una muestra poblacional con una franja amplia de
personas verdaderamente sanas, que es como Prati et al. (2002) y las guías
ACG/AASLD calibraron *sus* umbrales. Un clustering no supervisado busca la
separación más fuerte que exista en los datos, y en una cohorte
mayoritariamente enferma esa separación más fuerte es "**leve vs. severo**",
no "**sano vs. enfermo**". El grupo verdaderamente sano que el umbral de
literatura necesita como referencia está sub-representado en esta muestra
para que el clustering lo aísle de forma confiable.

**Esto refuerza, no contradice, la decisión ya tomada en el ADR 0004** de
adoptar umbrales de estudios poblacionales en vez de derivarlos de este
dataset — ahora hay evidencia empírica directa de por qué esa decisión fue
la correcta.

## Valor agregado — ¿el cluster "severo" apunta a distinta etiología?

El Experimento 1 sí encontró un cluster de severidad multivariada
consistente y separable. Como valor agregado, se revisa si ese grupo
también se distingue en el cociente De Ritis (AST/ALT) — el mismo indicio
**sugerente, no probatorio** de patrón de daño que ya se exploró en Loop C
(`docs/fuentes/Consulta_3.md`).

In [7]:
de_ritis_rows = []
for stratum in RNG_LABEL_ORDER:
    sub = exp1_results[stratum]["sub"].copy()
    sub["DeRitis"] = de_ritis_ratio(sub)
    tb_by_cluster = sub.groupby("Cluster")["TB"].median().sort_values(ascending=False)
    severe_cluster = tb_by_cluster.index[0]

    de_ritis_rows.append({
        "Estrato": stratum,
        "De_Ritis_mediana_severo": round(sub.loc[sub["Cluster"] == severe_cluster, "DeRitis"].median(), 2),
        "De_Ritis_mediana_resto": round(sub.loc[sub["Cluster"] != severe_cluster, "DeRitis"].median(), 2),
    })

pd.DataFrame(de_ritis_rows).set_index("Estrato")

,De_Ritis_mediana_severo,De_Ritis_mediana_resto
Estrato,,
0-17 · ambos sexos,1.33,1.14
18-39 · Female,1.11,0.91
18-39 · Male,1.28,1.20
40-59 · Female,1.22,1.19
40-59 · Male,1.98,1.07
60-120 · Female,1.90,1.21
60-120 · Male,1.35,1.29


**Interpretación.** En 5 de los 7 estratos la diferencia es chica
(menos de 0.15 puntos) — ahí la separación que encuentra el clustering es de
**magnitud** del daño (cuánto), no de **patrón** (qué tipo). Pero en 2
estratos la diferencia **no es chica**: `40-59 · Male` (1.98 vs. 1.07) y
`60-120 · Female` (1.90 vs. 1.21) — el cluster severo casi duplica el De
Ritis del resto, cruzando el punto de corte clásico de 2.0 asociado a
patrón alcohólico/cirrótico (`docs/fuentes/Consulta_3.md`). No es
sistemático entre estratos, así que **no se puede afirmar un patrón
general** — pero tampoco se puede descartar sin más. Es consistente con lo
que ya advertía Loop C: el De Ritis por sí solo, sin variable de etiología
para contrastar, no alcanza para probar nada. Se deja como hipótesis
abierta y específica (revisar `40-59 · Male` y `60-120 · Female` con más
detalle si se retoma este análisis), no como hallazgo cerrado.

## Caveat de honestidad metodológica — qué prueba esto y qué no

| Pregunta | Respuesta |
|---|---|
| ¿El clustering encuentra patrones reales? | **Sí** — un cluster de severidad multivariada consistente, separable en casi todos los estratos adultos. |
| ¿Ese patrón sugiere heterogeneidad de severidad/etiología? | **Sí, como hipótesis** — coincide con el eje de excreción biliar (`TB`) más que con un tipo etiológico específico. |
| ¿Produce puntos de corte diagnósticos utilizables? | **No.** Los rangos se superponen en la mayoría de los estratos; donde no se superponen, el corte resultante queda muy por encima de la literatura, por sesgo de muestreo (cohorte hospitalaria, no poblacional). |
| ¿Reemplaza los umbrales del ADR 0004? | **No, nunca.** Los umbrales de Prati/ACG/AASLD siguen siendo la referencia oficial del proyecto. |
| ¿`60-120 · Female` (n=20) es confiable? | **No** — clusters de 1-2 personas, ruido de muestra chica. Se reporta pero no se interpreta. |
| ¿Sirve para algo entonces? | Como *feature* candidato de severidad para una futura Fase 4+ de modelado (ver notas abajo), y como evidencia adicional para la hipótesis de sub-diagnóstico de Loop A (a completar comparando composición de clusters por sexo). |

## Notas para fases futuras (no se ejecuta nada aquí)

- **Fase 4+ (modelado, fuera de este PRD):** el cluster de severidad
  multivariada del Experimento 1 es candidato a *feature engineering*
  (p. ej. distancia al centroide del cluster severo del propio estrato
  edad-sexo) — mantiene la estratificación como referencia relativa en vez
  de comparar a cada paciente contra la población general.
- **Fase 5 (auditoría de fairness, fuera de este PRD):** falta comparar la
  composición por sexo de los clusters severos entre estratos — si mujeres
  con perfil "severo" están sub-representadas en el `Selector` positivo
  relativo a hombres con perfil similar, es evidencia adicional (no prueba)
  para la hipótesis de sub-diagnóstico ya registrada en
  `docs/CHANGELOG_iteraciones.md` (Loop A).
- Si en algún momento se consigue o construye una muestra con una franja
  amplia de personas sanas (fuera del alcance actual), repetir este
  clustering ahí sería la forma correcta de intentar derivar cortes
  data-driven — con esta cohorte hospitalaria no es posible, como quedó
  demostrado arriba.